## 1a. Indexing (Document Ingestion)

In [2]:
# !pip install youtube-transcript-api

In [3]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled

In [4]:
video_id = "EJ2gkL4D54A" # not URL, id only
try: 
    ytt = YouTubeTranscriptApi()
    transcript_list = ytt.fetch(video_id)
    # flatten to plain text
    # transcript = " ".join(chunk['text'] for chunk in transcript_list)
except TranscriptsDisabled:
    print("No captions available for this video.")

In [5]:
transcript = [transcript_list]

In [6]:
li = []
for snippet in transcript_list:
    # print(snippet.text)
    li.append(snippet.text)
    
transcript = " ".join(li)
transcript

'Most people are terrified of looking dumb. We want\xa0\nto sound smart, have the answer ready, and never\xa0\xa0 admit we don\'t know. But let me ask you, has that\xa0\never actually made your thinking sharper? Or has\xa0\xa0 it just made you defensive, anxious, and closed\xa0\noff to learning? Here\'s the paradox. The more\xa0\xa0 you try to act smart, the dumber your thinking\xa0\ngets. The more you allow yourself to act dumb,\xa0\xa0 the sharper your thinking becomes. Why? Because\xa0\nacting smart usually means protecting what you\xa0\xa0 already know. You cling to your opinions. You\xa0\ntalk more than you listen. You fill gaps with\xa0\xa0 assumptions so you don\'t look clueless. That shuts\xa0\ndown learning. But acting dumb flips the script.\xa0\xa0 It means admitting what you don\'t know, asking\xa0\nbasic questions, exploring without ego. And when\xa0\xa0 you do that, you open the door to real insight.\xa0\nPsychologists call this the beginner\'s mind. Kids\xa0\xa0 do this n

## 1b. Indexing (Text Splitting)

In [7]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [8]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [9]:
len(chunks)

6

In [10]:
chunks[0].page_content

"Most people are terrified of looking dumb. We want\xa0\nto sound smart, have the answer ready, and never\xa0\xa0 admit we don't know. But let me ask you, has that\xa0\never actually made your thinking sharper? Or has\xa0\xa0 it just made you defensive, anxious, and closed\xa0\noff to learning? Here's the paradox. The more\xa0\xa0 you try to act smart, the dumber your thinking\xa0\ngets. The more you allow yourself to act dumb,\xa0\xa0 the sharper your thinking becomes. Why? Because\xa0\nacting smart usually means protecting what you\xa0\xa0 already know. You cling to your opinions. You\xa0\ntalk more than you listen. You fill gaps with\xa0\xa0 assumptions so you don't look clueless. That shuts\xa0\ndown learning. But acting dumb flips the script.\xa0\xa0 It means admitting what you don't know, asking\xa0\nbasic questions, exploring without ego. And when\xa0\xa0 you do that, you open the door to real insight.\xa0\nPsychologists call this the beginner's mind. Kids\xa0\xa0 do this natura

## Step 1c and 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [11]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from dotenv import load_dotenv

In [12]:
load_dotenv()

True

In [13]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print(embeddings.embed_query("test"))

/tmp/ipykernel_12177/137272482.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
/home/sakshi-khatiwada/Desktop/My Repos/Python/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[0.011573448777198792, 0.025136185809969902, -0.03670185059309006, 0.05932485684752464, -0.00714902812615037, -0.04119424149394035, 0.0770873948931694, 0.03744254261255264, 0.012449026107788086, -0.006117618177086115, 0.017034240067005157, -0.07701537013053894, -0.00039415323408320546, 0.027909055352211, -0.0159891489893198, -0.06827525049448013, 0.008884689770638943, -0.020280729979276657, -0.08035990595817566, -0.013074049726128578, -0.041099995374679565, -0.025898084044456482, -0.0265386700630188, 0.03305231034755707, -0.022079166024923325, 0.021046139299869537, -0.05792199447751045, 0.03294876962900162, 0.029707415029406548, -0.06224840134382248, 0.03878802806138992, 0.03199075162410736, 0.015330808237195015, 0.0453069806098938, 0.053149450570344925, 0.013360696844756603, 0.04122491553425789, 0.028142863884568214, 0.019398430362343788, -0.003252319758757949, -0.003612351370975375, -0.1428602933883667, 0.03807118535041809, -0.010916211642324924, 0.026093997061252594, 0.0413699299097

In [15]:
# !pip install faiss-cpu

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Using cached faiss_cpu-1.12.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.1 kB)
Using cached faiss_cpu-1.12.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (31.4 MB)


In [16]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(chunks, embeddings)

In [17]:
vector_store.index_to_docstore_id

{0: '0073047c-8ace-41a3-8264-095565b7efc3',
 1: '06088869-3c46-4205-b8b8-28cd42bb6175',
 2: '89b20eed-fe54-4bea-9e7c-14005d5a7570',
 3: '6e5f873b-0332-41a3-88f4-088b66985cc6',
 4: '433ee4b9-0534-4375-a8eb-0d8cb4a6cb31',
 5: '7eded762-1663-4ba0-b152-066792188503'}

In [19]:
vector_store.get_by_ids(['433ee4b9-0534-4375-a8eb-0d8cb4a6cb31'])

[Document(id='433ee4b9-0534-4375-a8eb-0d8cb4a6cb31', metadata={}, page_content="You notice details you normally skip. You see\xa0\xa0 patterns others miss. You think from the ground\xa0\nup instead of layering guesses on guesses, and the\xa0\xa0 research backs it up. Studies on problem solving\xa0\nshow people with a beginner's mindset generate\xa0\xa0 more creative solutions than those relying only on\xa0\nexpertise. Expertise narrows thinking. Curiosity\xa0\xa0 expands it. Acting dumb forces curiosity. So,\xa0\nlet me ask you, what would happen if you stopped\xa0\xa0 pretending to be smart? What if you started asking\xa0\nthe questions everyone else is too scared to ask?\xa0\xa0 Here's the paradox. The smartest move you can make\xa0\nis to stop performing intelligence. Act dumb. Ask,\xa0\xa0 listen, experiment. That's how you cut through\xa0\nassumptions and actually think like a genius.\xa0\xa0 Because acting dumb doesn't mean you are dumb. It\xa0\nmeans you're bold enough to look p

## 2. Retriever

In [20]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 2})

In [21]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x79fb3be83470>, search_kwargs={'k': 2})

In [22]:
retriever.invoke("why we should act dumb?")

[Document(id='7eded762-1663-4ba0-b152-066792188503', metadata={}, page_content="assumptions and actually think like a genius.\xa0\xa0 Because acting dumb doesn't mean you are dumb. It\xa0\nmeans you're bold enough to look past appearances\xa0\xa0 and get to the truth. And truth, no matter how\xa0\nawkward or simple it looks, is always where genius\xa0\xa0 thinking starts. If you're still watching,\xa0\nlike and subscribe to support this channel."),
 Document(id='0073047c-8ace-41a3-8264-095565b7efc3', metadata={}, page_content="Most people are terrified of looking dumb. We want\xa0\nto sound smart, have the answer ready, and never\xa0\xa0 admit we don't know. But let me ask you, has that\xa0\never actually made your thinking sharper? Or has\xa0\xa0 it just made you defensive, anxious, and closed\xa0\noff to learning? Here's the paradox. The more\xa0\xa0 you try to act smart, the dumber your thinking\xa0\ngets. The more you allow yourself to act dumb,\xa0\xa0 the sharper your thinking be

## 3. Augmentation

In [99]:
llm = HuggingFaceEndpoint(
    repo_id="mistralai/Mistral-7B-Instruct-v0.3",
    task="text-generation",
)

model = ChatHuggingFace(llm=llm)

In [100]:
from langchain.prompts import PromptTemplate

In [101]:
prompt = PromptTemplate(
    template="""
    You are a helpful assistant. 
    Strictly Answer only from the provided transcript context.
    If the context is insufficient, just say you don't know.
    
    {context}
    Question: {question}""",
    input_variables=['context', 'question']
)

In [102]:
question = "why is it good to act dumb?"
retrieved_docs = retriever.invoke(question)

In [103]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)

In [104]:
context_text

"assumptions and actually think like a genius.\xa0\xa0 Because acting dumb doesn't mean you are dumb. It\xa0\nmeans you're bold enough to look past appearances\xa0\xa0 and get to the truth. And truth, no matter how\xa0\nawkward or simple it looks, is always where genius\xa0\xa0 thinking starts. If you're still watching,\xa0\nlike and subscribe to support this channel.\n\nMost people are terrified of looking dumb. We want\xa0\nto sound smart, have the answer ready, and never\xa0\xa0 admit we don't know. But let me ask you, has that\xa0\never actually made your thinking sharper? Or has\xa0\xa0 it just made you defensive, anxious, and closed\xa0\noff to learning? Here's the paradox. The more\xa0\xa0 you try to act smart, the dumber your thinking\xa0\ngets. The more you allow yourself to act dumb,\xa0\xa0 the sharper your thinking becomes. Why? Because\xa0\nacting smart usually means protecting what you\xa0\xa0 already know. You cling to your opinions. You\xa0\ntalk more than you listen. Y

In [105]:
final_prompt = prompt.invoke({"context": context_text, "question":question})

## 4. Generation

In [106]:
answer = model.invoke(final_prompt)
print(answer.content)


Acting dumb or being in a state of a beginner's mind is beneficial because it opens the door to learning. It allows you to let go of your preconceived notions and assumptions, opening yourself up to new perspectives and insights. This state of mind encourages curiosity and questioning, and fosters a growth mindset. Additionally, being humble and admitting what you don't know can build trust and rapport with others, making effective communication and collaboration more likely. Remember that it's okay not to have all the answers, and sometimes questioning what you think you know can lead to even greater understanding.

If you have any other questions, feel free to ask!




In [107]:
prompt2 = prompt.invoke({"context": context_text, "question": "How to be smartest of all?"})
answer2 = model.invoke(prompt2)
print(answer2)

content="\n    To be the smartest of all, it's essential to embrace a beginner's mind and continuous learning.\n    Being smart is not about always having the answers, but it's about the ability to learn,\n    adapt, and grow from new experiences. Below are some tips to cultivate the smartest version of yourself:\n\n    1. Ask questions: Ask questions to clarify your understanding and seek more information.\n    2. Embrace uncertainty: Admitting what you don't know can lead to unexpected insights and opportunities.\n    3. Be curious: Cultivate a deep curiosity about various topics and seek to learn and explore new things.\n    4. Learn from others: Surround yourself with people who have expertise in areas where you are not,\n       and be willing to listen and learn from them.\n    5. Practice critical thinking: Develop the ability to analyze information, think critically, and solve problems effectively.\n    6. Keep an open mind: An open mind allows you to consider alternative perspe

In [108]:
prompt2 = prompt.invoke({"context": context_text, "question": "what is cleaning?"})
answer2 = model.invoke(prompt2)
print(answer2)
#ERR - the llm was not supposed to answer this, it is still answering 

content="\nCleaning is the process of removing dust, debris, stains, or microorganisms from a surface in a process called housekeeping. It is usually done to maintain the cleanliness and sanitation of living spaces such as homes and offices, but also applies to other objects like cars and clothes.\n\nThe concept of cleaning has been present in various human societies throughout history, with methods and tools adopted according to cultural, technological, and historical context. For example, ancient civilizations like Egypt, Greece, and Rome had slaves or servants specifically dedicated to cleaning, while in the modern age, cleaning is often accomplished using cleaning products and specialized equipment.\n\nWhile cleaning is primarily a required task for maintaining hygiene and aesthetics, it can also impact the overall health and productivity of individuals. A clean environment can reduce the risk of illnesses caused by microorganisms, improve air quality, and reduce stress caused by c

## Chaining all of this

In [109]:
from langchain_core.output_parsers import StrOutputParser
from langchain.schema.runnable import RunnableParallel, RunnablePassthrough, RunnableLambda

In [110]:
parser = StrOutputParser()

In [117]:
def format_docs(retrieved_docs):
    context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text

In [118]:

question_chain = RunnableParallel({
    "question": RunnablePassthrough(),
    "context":  retriever | RunnableLambda(format_docs)
})

question_chain.invoke("what is what")

# final_chain = question_chain | prompt | llm | parser

{'question': 'what is what',
 'context': 'Psychologists call this the beginner\'s mind. Kids\xa0\xa0 do this naturally. They ask why 50 times in a row.\xa0\nIt\'s annoying, sure, but it\'s also why they learn\xa0\xa0 faster than adults. Adults stop doing it because\xa0\nwe\'re embarrassed to look dumb. But here\'s the\xa0\xa0 twist. The genius thinkers never stopped. Think\xa0\nabout the greatest discoveries in history. They\xa0\xa0 didn\'t start with someone flexing knowledge.\xa0\nThey started with dumb questions. Why does an\xa0\xa0 apple fall? Why does light bend? Why does the sky\xa0\nchange color? Simple questions, but they cracked\xa0\xa0 open entire fields of science. So, how do you use\xa0\nthis paradox in your own life? At work, someone\xa0\xa0 explains a process you don\'t fully get. Old move,\xa0\nnod along, pretend you understand, then struggle\xa0\xa0 later. New move, act dumb. Ask the obvious\xa0\nquestion. Can you walk me through that again?\xa0\xa0 For a second, you mi

In [119]:
final_chain.invoke("why act dumb?")

ValueError: Model mistralai/Mistral-7B-Instruct-v0.3 is not supported for task text-generation and provider novita. Supported task: conversational.